# Evaluation of fidelity and utility of synthetic data 
---
The goal of this notebook is to utilize a synthetic data generation tool to create synthetic tabular data. Afterward, the synthetic data’s fidelity and utility aspects should be evaluated. However, already existing tools should be leveraged and not implemented in any way. Therefore this notebook primarily involves leveraging existing code and implementing necessary boilerplate code to get some results.

## General data & imports
In this section we define globally used variables and import the most used packages.

In [ ]:
import sys
sys.path.append("../src")  # go to parent dir

import os
import warnings
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from models_utils import train_and_evaluate_pipeline
from synthetic_utils import generate_synthetic_data, evaluate_fidelity
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import run_diagnostic
from sdv.single_table import (
    GaussianCopulaSynthesizer,
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
)
from sdmetrics.reports.single_table import QualityReport
from sdmetrics.visualization import get_column_plot
from DataSynthesizer.lib.utils import display_bayesian_network
from DataSynthesizer.DataDescriber import DataDescriber
from DataSynthesizer.DataGenerator import DataGenerator

warnings.filterwarnings("ignore")

In [2]:
# define a random state
random_state = 12014500
np.random.seed(random_state)

## Dataset
In this notebook, we perform all action on the COMPASS-dataset. For our usecase we use the un-preprocessed version of it (./compass-scores-raw.csv). Therefore the following cells give a brief insight into the data itself.

In [ ]:
# fetch dataset
census_income = fetch_ucirepo(id=20)
df = census_income.data["original"]

In [4]:
display(df.head())

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
df["income"].value_counts()

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [6]:
df.shape

(48842, 15)

In [7]:
df.dtypes

age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
income            object
dtype: object

In [8]:
df.isna().sum()

age                 0
workclass         963
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation        966
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    274
income              0
dtype: int64

In [9]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


### PreProcessing
This sections deals, with the pre-processing we performed on the dataset.

In [ ]:
nominal_features = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
]
target = "income"

First, we dropped all features, that either had redundant information, like `education` being the textual version of `education-num`, or directly identifying informations like any kind of unique ids or names.

In [ ]:
columns_to_drop = ["education", "native-country"]
df_preprocessed = df.drop(columns_to_drop, axis=1)

Furthermore, we converted `sex` to binary to make calculations easier and converted. Also we unified `>50k` with `>50k.` and `<=50k` with `<=50k.` respectively of `income` into one class, since it has the same meaning. Finally, we converted it into a binary attribute

In [ ]:
df_preprocessed["sex"] = np.where((df_preprocessed["sex"] == "Male"), 1, 0)
df_preprocessed[target] = np.where(
    (df_preprocessed["income"].str.startswith(">50K")), 1, 0
)

Afterwards, we removed all records that had missing values in our label attribute `income`. Those records cannot be used in any way and therefore should be discarded.

In [13]:
df_preprocessed = df_preprocessed.dropna(subset=[target])

In [14]:
df_preprocessed.shape

(48842, 13)

#### Train/Test-Split
After this we perform the basic 80/20 split into train and test data.

In [ ]:
df_train, df_test = train_test_split(
    df_preprocessed, test_size=0.2, random_state=random_state
)

Lastly, we replaces all `NaN`-strings inside the train- and test-set individually with `np.nan` to make it more unified.

In [16]:
df_train.replace("NaN", np.nan, inplace=True)
df_test.replace("NaN", np.nan, inplace=True)

In [ ]:
df_train.to_json("../data/trainset.json")
df_test.to_json("../data/testset.json")

## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [19]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [21]:
metrics_orig = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.837
Precision       : 0.782
Recall          : 0.744
F1              : 0.760


## Data synthetization
This sections presents the creation and evaluation of our synthesized data. For this we use multiple instances from the [Synthetic Data Vault](https://github.com/sdv-dev/SDV) and two methods from [DataSynthesizer](https://github.com/DataResponsibly/DataSynthesizer). We only synthesize the training data, because we want to further evaluate on original test data.

### Synthetic Data Vault

In [ ]:
fit = False  # only use already generated data

#### Metadata
First, of all we need to create a metadata object of our training data. This simply is a representation of our data.

In [ ]:
os.makedirs("../data/SDV", exist_ok=True)
metadata = SingleTableMetadata()

try:
    metadata = metadata.load_from_json("./data/SDV/metadata.json")
except:
    metadata.detect_from_dataframe(df_train)
    metadata.save_to_json(filepath="./data/SDV/metadata.json")

Afterwards, we validate the metadata to ensure that it is correct.

In [25]:
metadata.validate_data(data=df_train)

#### GaussianCopulaSynthesizer
First we use the GaussianCopulaSynthesizer, to synthesize and save the dataset. Afterwards we evaluate some metrics on the dataset. For more concrete explanations and conclusions, please take a look at our report.

In [26]:
df_train_synth_gc = generate_synthetic_data(
    GaussianCopulaSynthesizer(metadata, enforce_min_max_values=True), df_train, fit=fit
)

In [ ]:
# save synthetic data for reproducibility
df_train_synth_gc.to_json("../data/synthetic_data_GaussianCopulaSynthesizer.json")

Afterwards, we run a diagnostic check, wether the data generation was successful.

In [28]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_gc, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 462.19it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 839.03it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [29]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_gc, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 90.74it/s]|
Column Shapes Score: 87.77%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:01<00:00, 39.66it/s]|
Column Pair Trends Score: 42.75%

Overall Score (Average): 65.26%



In [30]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,0.975354
1,workclass,TVComplement,0.997845
2,fnlwgt,KSComplement,0.957976
3,education-num,KSComplement,0.863026
4,marital-status,TVComplement,0.993013
5,occupation,TVComplement,0.990838
6,relationship,TVComplement,0.996673
7,race,TVComplement,0.997594
8,sex,TVComplement,0.997057
9,capital-gain,KSComplement,0.131779


In [31]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [32]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [33]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_gc,
    column_name="capital-gain",
    plot_type="distplot",
).show(renderer="vscode")

In [34]:
evaluate_fidelity(df_train, df_train_synth_gc)

{'CSTest': 0.9999999995420114,
 'ContinuousKLDivergence': 0.5706353286429244,
 'DiscreteKLDivergence': 0.8193866347995116,
 'LogisticDetection': 0.27159204775846446}

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [35]:
metrics_gc = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_gc,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.765
Precision       : 0.740
Recall          : 0.510
F1              : 0.456


#### CTGANSynthesizer
Second we use the CTGANSynthesizer, to synthesize and save the dataset. Afterwards we evaluate some metrics on the dataset. For more concrete explanations and conclusions, please take a look at our report.

In [ ]:
df_train_synth_ctgan = generate_synthetic_data(
    CTGANSynthesizer(metadata, enforce_min_max_values=True, epochs=250),
    df_train,
    fit=fit,
)

In [ ]:
df_train_synth_ctgan.to_json("../data/synthetic_data_CTGANSynthesizer.json")

Afterwards, we run a diagnostic check, wether the data generation was successful.

In [38]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_ctgan, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 463.56it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 544.93it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [39]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_ctgan, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 85.66it/s]|
Column Shapes Score: 87.3%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:02<00:00, 36.20it/s]|
Column Pair Trends Score: 41.96%

Overall Score (Average): 64.63%



In [40]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,0.934047
1,workclass,TVComplement,0.951564
2,fnlwgt,KSComplement,0.890308
3,education-num,KSComplement,0.924654
4,marital-status,TVComplement,0.889540
5,occupation,TVComplement,0.853838
6,relationship,TVComplement,0.921327
7,race,TVComplement,0.916003
8,sex,TVComplement,0.872239
9,capital-gain,KSComplement,0.478079


In [41]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [42]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [43]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ctgan,
    column_name="capital-gain",
    plot_type="distplot",
).show(renderer="vscode")

In [44]:
evaluate_fidelity(df_train, df_train_synth_ctgan)

{'CSTest': 0.9995130638115712,
 'ContinuousKLDivergence': 0.9371845260928924,
 'DiscreteKLDivergence': 0.8338417808492105,
 'LogisticDetection': 0.6550804758613773}

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [45]:
metrics_ctgan = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_ctgan,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.831
Precision       : 0.774
Recall          : 0.729
F1              : 0.747


#### TVAESynthesizer
Third we use the TVAESynthesizer, to synthesize and save the dataset. Afterwards we evaluate some metrics on the dataset. For more concrete explanations and conclusions, please take a look at our report.

In [46]:
df_train_synth_tvae = generate_synthetic_data(
    TVAESynthesizer(metadata, enforce_min_max_values=True, epochs=250),
    df_train,
    fit=fit,
)

In [ ]:
df_train_synth_tvae.to_json("../data/synthetic_data_TVAESynthesizer.json")

Afterwards, we run a diagnostic check, wether the data generation was successful.

In [48]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_tvae, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 447.44it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 1161.54it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [49]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_tvae, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 82.05it/s]|
Column Shapes Score: 89.5%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:02<00:00, 34.49it/s]|
Column Pair Trends Score: 43.02%

Overall Score (Average): 66.26%



In [50]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,0.894582
1,workclass,TVComplement,0.916469
2,fnlwgt,KSComplement,0.698718
3,education-num,KSComplement,0.938525
4,marital-status,TVComplement,0.920329
5,occupation,TVComplement,0.924807
6,relationship,TVComplement,0.925038
7,race,TVComplement,0.923400
8,sex,TVComplement,0.980319
9,capital-gain,KSComplement,0.622604


In [51]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [52]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [53]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_tvae,
    column_name="fnlwgt",
    plot_type="distplot",
).show(renderer="vscode")

In [54]:
evaluate_fidelity(df_train, df_train_synth_tvae)

{'CSTest': 0.9999319311931295,
 'ContinuousKLDivergence': 0.9107431907377007,
 'DiscreteKLDivergence': 0.8728108974089868,
 'LogisticDetection': 0.546821563925664}

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [55]:
metrics_tvae = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_tvae,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.820
Precision       : 0.752
Recall          : 0.753
F1              : 0.753


#### CopulaGANSynthesizer
Lastly we use the CopulaGANSynthesizer, to synthesize and save the dataset. Afterwards we evaluate some metrics on the dataset. For more concrete explanations and conclusions, please take a look at our report.

In [56]:
df_train_synth_cgan = generate_synthetic_data(
    CopulaGANSynthesizer(metadata, enforce_min_max_values=True, epochs=250),
    df_train,
    fit=fit,
)

In [ ]:
df_train_synth_cgan.to_json("../data/synthetic_data_CopulaGANSynthesizer.json")

Afterwards, we run a diagnostic check, wether the data generation was successful.

In [58]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_cgan, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 482.14it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 1387.46it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [59]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_cgan, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 73.17it/s]|
Column Shapes Score: 92.42%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:01<00:00, 42.10it/s]|
Column Pair Trends Score: 41.98%

Overall Score (Average): 67.2%



In [60]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,0.934712
1,workclass,TVComplement,0.901379
2,fnlwgt,KSComplement,0.946306
3,education-num,KSComplement,0.951347
4,marital-status,TVComplement,0.898933
5,occupation,TVComplement,0.888459
6,relationship,TVComplement,0.920201
7,race,TVComplement,0.833952
8,sex,TVComplement,0.959460
9,capital-gain,KSComplement,0.973741


In [61]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [62]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [63]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_cgan,
    column_name="capital-gain",
    plot_type="distplot",
).show(renderer="vscode")

In [64]:
evaluate_fidelity(df_train, df_train_synth_cgan)

{'CSTest': 0.9982564653753931,
 'ContinuousKLDivergence': 0.9473305203968131,
 'DiscreteKLDivergence': 0.8339697702677059,
 'LogisticDetection': 0.6219854193166163}

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [65]:
metrics_cgan = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_cgan,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.808
Precision       : 0.745
Recall          : 0.794
F1              : 0.761


### DataSynthesizer
Secondly, we implemented the DataSynthesizer, which is based on Bayesian Networks, to check how well this approach works on our sample data. For this we use two approaches. First we use independent-mode, which assumes all attributes are statistically indepentent to each other and second we use the correlation-mode which assumes non-independence between variables

In [67]:
# save data
dataset_file = "../data/trainset.csv"
df_train.to_csv(dataset_file, index=False)

threshold_value = 13

categorical_attributes = {
    col: True for col in df_train.columns if col in nominal_features
}
num_rows = len(df_train)

In [68]:
os.makedirs("../data/DataSynthesizer", exist_ok=True)

description_file = (
    f"../data/DataSynthesizer/independent_attribute_mode_description.json"
)
synthetic_data = (
    f"../data/synthetic_data_DataSynthesizer_independent_attribute_mode.csv"
)

describer = DataDescriber(category_threshold=threshold_value)
describer.describe_dataset_in_independent_attribute_mode(
    dataset_file=dataset_file,
    attribute_to_is_categorical=categorical_attributes,
    attribute_to_is_candidate_key={},
)
describer.save_dataset_description_to_file(description_file)

generator = DataGenerator()
generator.generate_dataset_in_independent_mode(num_rows, description_file)
generator.save_synthetic_data(synthetic_data)

df_train_synth_ds_independent = pd.read_csv(synthetic_data)

#### Independent attribute mode

Afterwards, we run a diagnostic check, wether the data generation was successful.

In [69]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_ds_independent, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 403.26it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 1358.70it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [70]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_ds_independent, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 67.11it/s]|
Column Shapes Score: 80.04%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:01<00:00, 40.86it/s]|
Column Pair Trends Score: 41.64%

Overall Score (Average): 60.84%



In [71]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,0.961124
1,workclass,TVComplement,0.965434
2,fnlwgt,KSComplement,0.956415
3,education-num,KSComplement,0.928390
4,marital-status,TVComplement,0.970747
5,occupation,TVComplement,0.949577
6,relationship,TVComplement,0.973793
7,race,TVComplement,0.968981
8,sex,TVComplement,0.999104
9,capital-gain,KSComplement,0.082487


In [72]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [73]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [74]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ds_independent,
    column_name="capital-gain",
    plot_type="distplot",
).show(renderer="vscode")

In [75]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ds_independent,
    column_name="capital-loss",
    plot_type="distplot",
).show(renderer="vscode")

In [76]:
evaluate_fidelity(df_train, df_train_synth_ds_independent)

{'CSTest': 0.9988145010567416,
 'ContinuousKLDivergence': 0.804346707729899,
 'DiscreteKLDivergence': 0.7634891787076071,
 'LogisticDetection': 0.3122776978790677}

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [77]:
metric_independent = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_ds_independent,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.762
Precision       : 0.381
Recall          : 0.500
F1              : 0.432


#### Correlated attribute mode

In [78]:
os.makedirs("../data/DataSynthesizer", exist_ok=True)

description_file = f"../data/DataSynthesizer/correlated_attribute_mode_description.json"
synthetic_data = f"../data/synthetic_data_DataSynthesizer_correlated_attribute_mode.csv"

describer = DataDescriber(category_threshold=threshold_value)
describer.describe_dataset_in_correlated_attribute_mode(
    dataset_file=dataset_file,
    epsilon=1,
    k=2,
    attribute_to_is_categorical=categorical_attributes,
    attribute_to_is_candidate_key={},
)
describer.save_dataset_description_to_file(description_file)

generator = DataGenerator()
generator.generate_dataset_in_correlated_attribute_mode(num_rows, description_file)
generator.save_synthetic_data(synthetic_data)

df_train_synth_ds_correlated = pd.read_csv(synthetic_data)

================ Constructing Bayesian Network (BN) ================
Adding ROOT relationship
Adding attribute marital-status
Adding attribute sex
Adding attribute age
Adding attribute income
Adding attribute occupation
Adding attribute workclass
Adding attribute education-num
Adding attribute hours-per-week
Adding attribute capital-loss
Adding attribute race
Adding attribute capital-gain
Adding attribute fnlwgt
========================== BN constructed ==========================


In [79]:
display_bayesian_network(describer.bayesian_network)

Constructed Bayesian network:
    marital-status has parents ['relationship'].
    sex            has parents ['marital-status', 'relationship'].
    age            has parents ['marital-status', 'relationship'].
    income         has parents ['age', 'relationship'].
    occupation     has parents ['age', 'sex'].
    workclass      has parents ['occupation', 'age'].
    education-num  has parents ['occupation', 'age'].
    hours-per-week has parents ['education-num', 'age'].
    capital-loss   has parents ['hours-per-week', 'income'].
    race           has parents ['hours-per-week', 'age'].
    capital-gain   has parents ['education-num', 'income'].
    fnlwgt         has parents ['capital-loss', 'age'].


Afterwards, we run a diagnostic check, wether the data generation was successful.

In [80]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_ds_correlated, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 418.07it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 1347.35it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [81]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_ds_correlated, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 75.70it/s]|
Column Shapes Score: 75.92%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:02<00:00, 37.52it/s]|
Column Pair Trends Score: 40.63%

Overall Score (Average): 58.28%



In [82]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,0.968597
1,workclass,TVComplement,0.833632
2,fnlwgt,KSComplement,0.738336
3,education-num,KSComplement,0.824226
4,marital-status,TVComplement,0.991605
5,occupation,TVComplement,0.953808
6,relationship,TVComplement,0.992399
7,race,TVComplement,0.720881
8,sex,TVComplement,0.999616
9,capital-gain,KSComplement,0.082512


In [83]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [84]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [85]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ds_correlated,
    column_name="capital-gain",
    plot_type="distplot",
).show(renderer="vscode")

In [86]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ds_correlated,
    column_name="capital-loss",
    plot_type="distplot",
).show(renderer="vscode")

In [87]:
evaluate_fidelity(df_train, df_train_synth_ds_correlated)

{'CSTest': 0.8387234201313947,
 'ContinuousKLDivergence': 0.5008312835482046,
 'DiscreteKLDivergence': 0.6771832456650564,
 'LogisticDetection': 0.22746513380929478}

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [88]:
metric_correlated = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_ds_correlated,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.770
Precision       : 0.698
Recall          : 0.537
F1              : 0.513


#### Random Mode

In [89]:
os.makedirs("../data/DataSynthesizer", exist_ok=True)

mode = "random_mode"
description_file = f"../data/DataSynthesizer/random_mode_description.json"
synthetic_data = f"../data/synthetic_data_DataSynthesizer_random_mode_mode.csv"

describer = DataDescriber(category_threshold=threshold_value)
describer.describe_dataset_in_random_mode(dataset_file)
describer.save_dataset_description_to_file(description_file)

generator = DataGenerator()
generator.generate_dataset_in_random_mode(num_rows, description_file)
generator.save_synthetic_data(synthetic_data)

df_train_synth_ds_random = pd.read_csv(synthetic_data)

Afterwards, we run a diagnostic check, wether the data generation was successful.

In [90]:
diagnostic_report = run_diagnostic(
    real_data=df_train, synthetic_data=df_train_synth_ds_random, metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 13/13 [00:00<00:00, 447.83it/s]|
Data Validity Score: 75.95%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 843.25it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 87.98%



##### Fidelity
Now, we check for the fidelity by checking how similiar the synthetic data is in comparison to the original one.

In [91]:
quality_report = QualityReport()
quality_report.generate(df_train, df_train_synth_ds_random, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 13/13 [00:00<00:00, 125.40it/s]|
Column Shapes Score: 39.28%

(2/2) Evaluating Column Pair Trends: |██████████| 78/78 [00:02<00:00, 36.72it/s]|
Column Pair Trends Score: 26.36%

Overall Score (Average): 32.82%



In [92]:
quality_report.get_details(property_name="Column Shapes")

,Column,Metric,Score
0,age,KSComplement,6.773219e-01
1,workclass,TVComplement,4.024557e-01
2,fnlwgt,KSComplement,0.000000e+00
3,education-num,KSComplement,1.705014e-01
4,marital-status,TVComplement,4.990403e-01
5,occupation,TVComplement,6.786918e-10
6,relationship,TVComplement,6.721265e-01
7,race,TVComplement,3.478105e-01
8,sex,TVComplement,8.335168e-01
9,capital-gain,KSComplement,9.305659e-02


In [93]:
quality_report.get_visualization(property_name="Column Pair Trends").show(
    renderer="vscode"
)

In [94]:
quality_report.get_visualization(property_name="Column Shapes").show(renderer="vscode")

In [95]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ds_random,
    column_name="capital-gain",
    plot_type="distplot",
).show(renderer="vscode")

In [96]:
get_column_plot(
    real_data=df_train,
    synthetic_data=df_train_synth_ds_random,
    column_name="capital-loss",
    plot_type="distplot",
).show(renderer="vscode")

In [97]:
# since we generate randomly, we can't evaluate the fidelity
# evaluate_fidelity(df_train, synthetic_data_random_mode)

##### Utility
Secondly, we look how well our classifier performs on this newly generated dataset.

In [98]:
metric_random = train_and_evaluate_pipeline(
    clf,
    nominal_features,
    df_train_synth_ds_random,
    df_test,
    target,
    drop_na=True,
    verbose=True,
)

Metric          : Value          
Accuracy        : 0.238
Precision       : 0.119
Recall          : 0.500
F1              : 0.192


## Summary
Lastly, we summarize the resulting metrics

In [99]:
df = pd.DataFrame(
    [
        metrics_orig,
        metrics_gc,
        metrics_ctgan,
        metrics_tvae,
        metrics_cgan,
        metric_independent,
        metric_correlated,
        metric_random,
    ]
)
df.index = [
    "Original",
    "GaussianCopula",
    "CTGAN",
    "TVAE",
    "CopulaGAN",
    "Independent",
    "Correlated",
    "Random",
]

df

,Accuracy,Precision,Recall,F1
Original,0.837036,0.782159,0.744208,0.759739
GaussianCopula,0.765483,0.740088,0.510356,0.456063
CTGAN,0.830587,0.773966,0.729173,0.746604
TVAE,0.820452,0.752413,0.753010,0.752711
CopulaGAN,0.807657,0.744997,0.793895,0.760846
Independent,0.762105,0.381052,0.500000,0.432497
Correlated,0.770499,0.698246,0.536732,0.512771
Random,0.237895,0.118948,0.500000,0.192177
